## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [2]:
# Se agrega el código extraído del notebook en el que se enuncia el desafío y que se usó para resolverlo.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

from sklearn.datasets import fetch_20newsgroups
import numpy as np

In [3]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

tfidfvect = TfidfVectorizer()

X_train = tfidfvect.fit_transform(newsgroups_train.data)

print(type(X_train))
print(f'X_train shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

y_train = newsgroups_train.target
print(f'y_train shape: {y_train.shape}')

X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}


<class 'scipy.sparse._csr.csr_matrix'>
X_train shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631
y_train shape: (11314,)


##### Consigna 1

In [4]:

# === Consigna 1 ===================================================================
# Primero se revisa la cantidad de documentos y luego se generan 5 enteros aleatorios
# entre 0 y el valor obtenido.
total_docs = X_train.shape[0]

# rng = np.random.default_rng(seed=15)
# random_docs = rng.integers(low=0, high=total_docs, size=5)
random_docs = [10535,  7837,  7982,  9230,  2734] # Se dejan hardcodeados

for doc_idx in random_docs:
    print("===================================================================")
    print(f"Documento {doc_idx} | Tipo: {newsgroups_train.target_names[y_train[doc_idx]]}")
    print(newsgroups_train.data[doc_idx])
    cossim = cosine_similarity(X_train[doc_idx], X_train)[0]
    most5sim = np.argsort(cossim)[::-1][1:6]
    print("Similares:")
    for sim in most5sim:
        print(f"===> Doc {sim} | {newsgroups_train.target_names[y_train[sim]]}")
        # print(newsgroups_train.data[sim])

Documento 10535 | Tipo: talk.politics.guns





No, they didn't have electrical power, but no, I don't find the idea of
Davidians calmly cooking lunch with gas masks on as the FBI knocks the
buildings down very credible,either.

It's not like this whole discussion is relevant.  It started when some-
one made the wholly unsubstantiated allegation that the wood stove ig-
nited NAPALM the FBI shot into the buildings.

I'm not a groveling apoligist for the feds, far from it.  But wild ac-
cusations like this are ridiculous and obfuscate legitimate criticism of
their conduct in this whole affair.
Similares:
===> Doc 5895 | talk.politics.guns
===> Doc 2794 | talk.politics.guns
===> Doc 6080 | talk.politics.guns
===> Doc 6894 | talk.politics.guns
===> Doc 649 | talk.religion.misc
Documento 7837 | Tipo: misc.forsale
OK... I've done a little research and the price I've been asking
was a BIT high.  So...

	Casio CZ-101 Synthesizer	$125 or best offer

Features:
uses FM modulation to create sounds

Análisis

El documento 10535 tiene similares varios documentos de su mismo tipo **talk.politics.guns**, y tiene sentido por la mención de palabras como FBI, gas masks, feds and NAPALM. Solo parece raro el 5to documento más similar, que es de tipo **talk.religin.misc**, pero revisando el texto se encuentra que aparecen palabras en comun, como FBI.

El documento 7837, de tipo **misc.forsale**, no coincide con ninguno de sus 5 más similares, que son principalmente de los tipos **comp.os.ms-windows.misc** y **comp.sys.mac.hardware**. El error es entendible, ya que el documento principal refiere a una venta de un sintetizador Casio, cuya descripción de atributos puede asemejarse bastante a los de computadoras.

Para el documento 7982, de tipo **rec.autos**, únicamente coincide con el 5to más similar (Doc 2697). Los demás son de tipos bastante variados, todos distintos, probablemente porque el documento principal parece ser un mensaje de respuesta bastante informal.

El documento 9230 es de tipo **rec.sport.hockey** y no coincide con ninguno de los 5 más similares, entre los cuales aparece 3 veces el tipo **talk.politics.mideast**, quizás porque parece un análisis de un jugador y se mencionan varios nombres poco comunes. Deberían revisarse en profundidad todos los documentos para definirlo.

Para el documento 2734, su tipo (**comp.sys.mac.hardware**) solo coincide con el 5to más similar nuevamente, sin embargo los demás documentos son de un tipo parecido: **comp.sys.ibm.pc.hardware**. Tiene sentido ya que en el documento principal no se menciona nada específico de MAC, solo de computadoras.

##### Consigna 2

In [ ]:
# Consigna 2

# Se hace un clasificador zero-shot, tal que al documento a 
# clasificar solo se le asigne el tipo del documento más similar
# (según cosine similarity) del dataset de entrenamiento.
# Se obtiene la vectorizacion para todo test, se obtiene la
# matriz de similitudes con train, y para cada test se extrae
# el más similar.

X_test = tfidfvect.transform(newsgroups_test.data)

sim_matrix = cosine_similarity(X_test, X_train)
best_train_idx = np.argmax(sim_matrix, axis=1)
# y_pred_zs = np.array(
#     [newsgroups_train.target[y_train[i]] for i in best_train_idx],
#     dtype=int)
y_pred_zs = y_train[best_train_idx]
# pred_zero_shot = [newsgroups_train.target_names[y_train[i]] for i in best_train_idx]

print("Reales:      ", y_test[:20])
print("Predichos:   ", y_pred_zs[:20])
print("F1-Score (Macro) > ", f1_score(y_test, y_pred_zs, average='macro'))


Reales:       [ 7  5  0 17 19 13 15 15  5  1  2  5 17  8  0  2  4  1  6 16]
Predichos:    [ 7  5  7  7  7 14  1 13 16  4  4  4  7  3  7  4 14  4  7  1]
F1-Score (Macro) >  0.018386894900551766


El resultado, segun la métrica F1-Score (macro), es muy malo, pero ya se intuía de la consigna 1 que podía ser así. El documento más similar casi nunca es del mismo tipo que el documento a predecir.

##### Consigna 3

Los parámetros son los mismos para ambas funciones, pero ComplementNB además agrega el parámetro `norm`.

La documentación indica que para la clase `MultinomialNB`, el parámetro `class_prior` se usa cuando la probabilidad a priori real no se ajusta a la del dataset. Como se desconoce, es un parámetro que no se configurará.

Para la clase `ComplementNB`, la documentación indica que `fit_prior` solo se utiliza in casos edge con una sola clase en el dataset y que `class_prior` directamente no se usa, por lo cual ambos se excluyen.

El parámetro `force_alpha` solo setea `alpha` a 1e-10 si es menor a ese valor, para evitar errores numéricos. 

Para los demás parámetros se configura una búsqueda de grilla para explorar como afectan a los resultados.

In [6]:
# Se prueban distintas combinaciones de parámetros median un GridSearch
from sklearn.model_selection import GridSearchCV
import pandas as pd

multinomial_param_grid = {
    "alpha": [0.001, 0.01, 0.1, 0.5, 1.0],
    # "force_alpha": [True, False],
    "fit_prior": [True, False]
}

multinomial_grid = GridSearchCV(
    estimator=MultinomialNB(),
    param_grid=multinomial_param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2
)

multinomial_grid.fit(X_train, y_train)

print(multinomial_grid.best_params_)
print(multinomial_grid.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
{'alpha': 0.01, 'fit_prior': False}
0.7601998967577746


In [7]:
multinomial_results = pd.DataFrame(multinomial_grid.cv_results_)
multinomial_results = multinomial_results.sort_values("rank_test_score")

print(multinomial_results[
    [
        "mean_test_score",
        "std_test_score",
        "param_alpha",
        # "param_force_alpha",
        "param_fit_prior"
    ]
])

   mean_test_score  std_test_score  param_alpha  param_fit_prior
3         0.760200        0.008294        0.010            False
2         0.756031        0.008756        0.010             True
1         0.750070        0.008158        0.001            False
0         0.748831        0.007736        0.001             True
5         0.727905        0.005509        0.100            False
4         0.718850        0.005923        0.100             True
7         0.676466        0.008110        0.500            False
6         0.662695        0.007387        0.500             True
9         0.639744        0.008519        1.000            False
8         0.619959        0.011484        1.000             True


In [8]:
complement_param_grid = {
    "alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
    # "force_alpha": [True, False],
    "norm": [True, False]
}

complement_grid = GridSearchCV(
    estimator=ComplementNB(),
    param_grid=complement_param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2
)

complement_grid.fit(X_train, y_train)

print(complement_grid.best_params_)
print(complement_grid.best_score_)


Fitting 5 folds for each of 10 candidates, totalling 50 fits
{'alpha': 0.1, 'norm': False}
0.7656097292010164


In [9]:
complement_results = pd.DataFrame(complement_grid.cv_results_)
complement_results = complement_results.sort_values("rank_test_score")

print(complement_results[
    [
        "mean_test_score",
        "std_test_score",
        "param_alpha",
        # "param_force_alpha",
        "param_norm"
    ]
])

   mean_test_score  std_test_score  param_alpha  param_norm
3         0.765610        0.007240         0.10       False
2         0.762938        0.008088         0.10        True
5         0.760012        0.008351         0.50       False
4         0.753670        0.008659         0.50        True
7         0.751295        0.010250         1.00       False
6         0.745356        0.011109         1.00        True
1         0.742247        0.006785         0.01       False
0         0.739518        0.005718         0.01        True
9         0.736043        0.009336         2.00       False
8         0.732484        0.009164         2.00        True


In [10]:
# Nos quedamos con los mejores parámetros para cada uno y los
# comparamos sobre el dataset de test.

multinomial_best = multinomial_grid.best_estimator_
y_pred_mult = multinomial_best.predict(X_test)
f1_score_mult = f1_score(y_test, y_pred_mult, average="macro")
print(f"F1-Score MultinomialNB --> {f1_score_mult}")


complement_best = complement_grid.best_estimator_
y_pred_comp = complement_best.predict(X_test)
f1_score_comp = f1_score(y_test, y_pred_comp, average="macro")
print(f"F1-Score ComplementNB --> {f1_score_comp}")

F1-Score MultinomialNB --> 0.6895480073763225
F1-Score ComplementNB --> 0.6953652590540836


In [11]:
print(y_pred_mult[:20])
print(y_pred_comp[:20])
print(y_test[:20])

[ 7  1 19 17  0 13 15  2  5  1  2  5 17  8 15  3  1  3  4 16]
[ 1  1 15 17  0 13 15 12  5  1  2  5 17  8 15  3  4  3  3 16]
[ 7  5  0 17 19 13 15 15  5  1  2  5 17  8  0  2  4  1  6 16]


In [12]:
np.unique(y_train, return_counts=True)

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19]),
 array([480, 584, 591, 590, 578, 593, 585, 594, 598, 597, 600, 595, 591,
        594, 593, 599, 546, 564, 465, 377]))

Los mejores parámetros son:
- MultinomialNB
    - alpha: 0.01
    - fit_prior: False
- ComplementNB
    - alpha: 0.10
    - norm: False

Los resultados son similares en general: el F1-Score (Macro) es de **0.6895** para el clasificador Multinomial y **0.6954** para el Complement. Según la documentación, ComplementNB fue diseñado para corregir algunos supuestos del Multinomial y que es particularmente adecuado para datasets desbalanceados; y este dataset no es ese caso, por lo que ese podría ser el motivo de que finalmente tengan un desempeño bastante parecido.

##### Consigna 4

In [13]:
# Matriz término documento
X_term_doc = X_train.T

# Esto intentó generar una matriz de 77 GB!
# term_sim = cosine_similarity(X_term_doc)

# Se busca la similitud solo con las 5 palabras elegidas
words = ["car", "money", "god", "mouse", "star"]
indices = [tfidfvect.vocabulary_[w] for w in words]
subset = X_term_doc[indices]

word_sim = cosine_similarity(subset, X_term_doc)

for i, w in enumerate(words):
    top5 = np.argsort(word_sim[i])[::-1][1:6]
    print(f"{w} -> {[idx2word[j] for j in top5]}")




car -> ['cars', 'criterium', 'civic', 'owner', 'dealer']
money -> ['buyback', 'computerize', 'fundraising', 'to', 'spend']
god -> ['jesus', 'bible', 'that', 'existence', 'christ']
mouse -> ['com1', 'jumpiness', 'cacheing', 'minesweeper', 'shortcut']
star -> ['trek', 'dragonslayer', 'extratresstials', '2061', 'compleat']


A partir de las 5 palabras elegidas, se opbtuvieron las siguientes mayores similitudes:
- car -> cars, criterium, civic, owner, dealer
- money -> buyback, computerize, fundraising, to, spend
- god -> jesus, bible, that, existence, christ
- mouse -> com1, jumpiness, cacheing, minesweeper, shortcut
- star -> trek, dragonslayer, extratresstials, 2061, compleat

Y se podría decir que en casi todos los casos tiene sentido que sean palabras que aparezcan en el mismo documento que sus respectivas palabras originales.